In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import torch
import itertools

# 1. Load your spatial data
lakes = gpd.read_file(r"model_data\Final data\stage2_dataset_15.gpkg")
lakes2 = gpd.read_file(r"model_data\Final data\stage_2_final_dataset5.gpkg")

In [8]:
lakes.columns.tolist()

['original_row_id',
 'Type',
 'Elevation',
 'GLAKE_ID',
 'area_2020',
 'perimeter_2020',
 'area_1990',
 'perimeter_1990',
 'geometry_1990',
 'area_2000',
 'perimeter_2000',
 'geometry_2000',
 'area_2010',
 'perimeter_2010',
 'geometry_2010',
 'area_2015',
 'perimeter_2015',
 'geometry_2015',
 'area_1990_km2',
 'area_2000_km2',
 'area_2010_km2',
 'area_2015_km2',
 'area_2020_km2',
 'expansion_rate_km2_yr',
 'expansion_rate_pct_yr',
 'glof_happened',
 'glof_count',
 'lake_type',
 'distance_to_glacier_m',
 'nearest_rgiid',
 'is_connected',
 'wse_m_dam',
 'eq_susceptibility',
 'ls_susceptibility',
 'volume_m3',
 'max_depth_m',
 'surface_elevation_m',
 'area_m2',
 'perimeter_m',
 'RGIId',
 'G11_mean_slope_deg',
 'area_exp_1990_2000',
 'area_exp_pct_1990_2000',
 'area_cagr_pct_1990_2000',
 'area_exp_2000_2010',
 'area_exp_pct_2000_2010',
 'area_cagr_pct_2000_2010',
 'area_exp_2010_2015',
 'area_exp_pct_2010_2015',
 'area_cagr_pct_2010_2015',
 'area_exp_2015_2020',
 'area_exp_pct_2015_2020',


In [2]:
catchments = gpd.read_file(r"C:\Users\rubel\Downloads\GRITv1.0_segment_catchments_AS_EPSG4326.gpkg\GRITv1.0_segment_catchments_AS_EPSG4326.gpkg")
catchments.columns

Index(['cat', 'global_id', 'catchment_id', 'area', 'domain', 'geometry'], dtype='object')

In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np
import torch
import itertools

# 1. Load your spatial data
# lakes = gpd.read_file(r"model_data\Final data\trial_1064_full_dataset_susceptibility.gpkg")
# catchments = gpd.read_file(r"C:\Users\rubel\Downloads\GRITv1.0_segment_catchments_AS_EPSG4326.gpkg\GRITv1.0_segment_catchments_AS_EPSG4326.gpkg")

# Ensure CRS (Coordinate Reference System) matches before joining
catchments = catchments.to_crs(lakes.crs)

# 2. Spatial Join: Assign each lake a Catchment ID
# 'how="inner"' drops lakes that fall outside your defined catchments
# 'predicate="within"' ensures the lake centroid is inside the polygon
lakes['geometry'] = lakes.centroid # Use centroids for the join
lakes_with_basins = gpd.sjoin(lakes, catchments, how="inner", predicate="within")

# Check to ensure your lakes have an assigned 'catchment_id' (or equivalent column from your shapefile)
print(lakes_with_basins[['GLAKE_ID', 'catchment_id']].head())

          GLAKE_ID  catchment_id
0  GL077099E35395N     170000046
1  GL078223E34278N     170000046
2  GL078079E34401N     170000046
3  GL078069E34414N     170000046
4  GL078100E34520N     170000046


In [4]:
lakes_with_basins.to_file(r"model_data\Final data\lakes_with_catchments.gpkg", driver="GPKG")

In [6]:
lakes.shape

(6922, 89)

In [5]:
lakes_with_basins.shape

(6907, 95)

In [19]:
missing_lakes = lakes[~lakes['GLAKE_ID'].isin(lakes_with_basins['GLAKE_ID'])]
missing_lakes['GLAKE_ID'].values

array(['GL081474E30482N', 'GL081481E30461N', 'GL085460E28681N',
       'GL085450E28714N', 'GL085454E28716N', 'GL085441E28720N',
       'GL090875E30573N', 'GL090516E30434N', 'GL090496E30425N',
       'GL090501E30423N', 'GL090413E30413N', 'GL090423E30405N',
       'GL090740E30529N', 'GL090749E30513N', 'GL090421E30409N'],
      dtype=object)

In [5]:
lakes_with_basins = lakes_with_basins.reset_index(drop=True)

In [9]:
# 1. Reset index so DataFrame row numbers exactly match PyTorch Geometric node indices (0 to N-1)
lakes_with_basins = lakes_with_basins.reset_index(drop=True)

# 2. Save a mapping file of Node Index -> GLAKE_ID and era5_fid so you can match them later
mapping_df = lakes_with_basins[['GLAKE_ID', 'era5_fid']].copy()
mapping_df.index.name = 'node_index'
mapping_df.to_csv("gnn/node_mapping.csv")
print(f"Saved mapping to 'gnn/node_mapping.csv'. Example mapping: Node 0 -> {mapping_df.iloc[0]['GLAKE_ID']} (era5_fid: {mapping_df.iloc[0]['era5_fid']})")

# 3. Use the row-index approach to build the full directed graph.
# Each higher-elevation lake connects to every lower-elevation lake in the same catchment.
edges_source = []
edges_target = []

for basin_id, group in lakes_with_basins.groupby('catchment_id'):
    lake_indices = group.index.to_numpy()
    if len(lake_indices) <= 1:
        continue
    elevations = group['Elevation'].to_numpy()
    
    for local_i, node_i in enumerate(lake_indices):
        lower_candidates = np.flatnonzero(elevations[local_i] > elevations)
        if lower_candidates.size == 0:
            continue
        
        edges_source.extend([node_i] * lower_candidates.size)
        edges_target.extend(lake_indices[lower_candidates])

# Directed PyTorch edge_index
edge_index = torch.tensor([edges_source, edges_target], dtype=torch.long)
print(f"PyTorch edge_index shape: {edge_index.shape}")

Saved mapping to 'gnn/node_mapping.csv'. Example mapping: Node 0 -> GL077099E35395N (era5_fid: 1)
PyTorch edge_index shape: torch.Size([2, 9595114])


In [10]:
edge_index.shape

torch.Size([2, 9595114])

In [7]:
import networkx as nx

# Create a NetworkX Directed Graph using the edge_index
G = nx.DiGraph()

# Convert tensor to numpy arrays to extract edges
sources = edge_index[0].numpy()
targets = edge_index[1].numpy()

# Add edges to the NetworkX graph
G.add_edges_from(zip(sources, targets))

print(f"NetworkX Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

NetworkX Graph created with 6907 nodes and 9595114 edges


In [11]:
import torch_geometric

In [12]:
import torch

# Save the PyTorch tensor to a file
EDGE_INDEX_PATH = r"gnn\edge_index3.pt"
torch.save(edge_index, EDGE_INDEX_PATH)
print(f"edge_index saved successfully to {EDGE_INDEX_PATH}")

edge_index saved successfully to edge_index.pt


In [1]:
import torch

# Load the saved edge index (it's inside the 'gnn' folder)
EDGE_INDEX_PATH = r"gnn\edge_index3.pt"
try:
    edge_index = torch.load(EDGE_INDEX_PATH, map_location='cpu')
    print(f"Successfully loaded edge_index from '{EDGE_INDEX_PATH}'")
except FileNotFoundError:
    print(f"{EDGE_INDEX_PATH} not found. Make sure the file exists.")

try:
    from torch_geometric.data import Data
    
    # Create the graph using torch_geometric
    num_nodes = int(edge_index.max().item()) + 1 if edge_index.numel() else 0
    graph_data = Data(edge_index=edge_index, num_nodes=num_nodes)
    
    print("\nGraph built successfully with PyTorch Geometric!")
    print(f"Number of nodes: {graph_data.num_nodes}")
    print(f"Number of edges: {graph_data.num_edges}")
    print("Graph details:", graph_data)
except Exception as e:
    print("\nError importing torch_geometric or building graph:", e)

Successfully loaded edge_index from 'gnn\edge_index3.pt'

Graph built successfully with PyTorch Geometric!
Number of nodes: 6907
Number of edges: 25094
Graph details: Data(edge_index=[2, 25094], num_nodes=6907)


### Faster Epochs with Window Sampling

The original dataset uses every daily sliding window, so adjacent samples overlap heavily. For example, the windows starting at day 0 and day 1 share 92 out of 93 timesteps. This cell keeps the model and graph unchanged, but trains each epoch on a smaller set of window starts.

- `WINDOW_STRIDE = 7` uses weekly-spaced windows: 0, 7, 14, ...
- `MAX_BATCHES_PER_EPOCH = 1000` limits one epoch to about 1000 optimizer steps.
- `SHUFFLE_WINDOWS = True` picks a different sampled order each epoch.

Increase `MAX_BATCHES_PER_EPOCH` for more training coverage, or set it to `None` to use every strided window.

In [9]:
import os
import numpy as np
import torch

WINDOW_STRIDE = 7
MAX_BATCHES_PER_EPOCH = 1500
SHUFFLE_WINDOWS = True
SAMPLING_SEED = 42

class SampledGLOFParquetDataset(GLOFParquetDataset):
    def __init__(self, *args, window_stride=1, max_windows_per_epoch=None, shuffle_windows=True, seed=42, **kwargs):
        super().__init__(*args, **kwargs)
        self.window_stride = max(1, int(window_stride))
        self.max_windows_per_epoch = None if max_windows_per_epoch is None else int(max_windows_per_epoch)
        self.shuffle_windows = bool(shuffle_windows)
        self.seed = int(seed)
        self.epoch_counter = 0

    def _ensure_buffer_ready(self):
        if getattr(self, 'buffer', None) is not None:
            return

        if self.cache_path and os.path.exists(self.cache_path):
            print(f"Loading cached ERA5 buffer from {self.cache_path}...")
            self.buffer = np.load(self.cache_path, mmap_mode='r')
            self.total_days = self.buffer.shape[0]
            print(f"Cached buffer ready: {self.buffer.shape}")
            return

        print(f"Loading full Parquet dataset from {self.parquet_path}...")
        time_col = 'date'

        import pyarrow.parquet as pq
        import gc

        req_cols = ['fid', time_col] + self.feature_cols
        pf = pq.ParquetFile(self.parquet_path)

        print("Scanning dataset in chunks to find unique dates (saving RAM)...")
        unique_dates = set()
        for batch in pf.iter_batches(columns=[time_col]):
            unique_dates.update(batch.column(0).to_pandas().unique())

        unique_dates = np.sort(list(unique_dates))
        total_days = len(unique_dates)
        date_to_idx = {d: i for i, d in enumerate(unique_dates)}

        print(f"Allocating precise 3D Numpy buffer for {total_days} days and {self.num_nodes} nodes...")
        buffer = np.zeros((total_days, self.num_nodes, len(self.feature_cols)), dtype=np.float32)

        print("Populating buffer chunk-by-chunk to prevent memory overflow...")
        for chunk_idx, batch in enumerate(pf.iter_batches(columns=req_cols, batch_size=2000000)):
            df_chunk = batch.to_pandas()
            df_chunk = df_chunk[df_chunk['fid'].isin(self.valid_fids)]
            if df_chunk.empty:
                continue

            df_chunk['node_idx'] = df_chunk['fid'].map(self.node_to_idx)
            df_chunk['time_idx'] = df_chunk[time_col].map(date_to_idx)

            time_indices = df_chunk['time_idx'].values.astype(int)
            node_indices = df_chunk['node_idx'].values.astype(int)
            feature_values = df_chunk[self.feature_cols].values

            buffer[time_indices, node_indices, :] = feature_values

            del df_chunk, time_indices, node_indices, feature_values
            if chunk_idx % 10 == 0:
                gc.collect()
            if chunk_idx % 5 == 0:
                print(f"  Processed ~{chunk_idx * 2} million rows...")

        if self.cache_path:
            os.makedirs(os.path.dirname(self.cache_path) or '.', exist_ok=True)
            print(f"Saving ERA5 buffer cache to {self.cache_path}...")
            np.save(self.cache_path, buffer)

        self.buffer = buffer
        self.total_days = total_days
        print("Buffer perfectly prepared without crashing RAM!")

    def _make_sample(self, start_t):
        window = self.buffer[start_t : start_t + self.seq_len]

        mean = np.nanmean(window[:self.hist_len], axis=0, keepdims=True)
        std = np.nanstd(window[:self.hist_len], axis=0, keepdims=True) + 1e-8
        normalized_window = (window - mean) / std

        history = normalized_window[:self.hist_len]
        forecast_drivers = normalized_window[self.hist_len:]

        forecast_input = np.copy(forecast_drivers)
        forecast_input[:, :, 6:9] = 0.0

        x_seq = np.concatenate([history, forecast_input], axis=0)
        y_seq = window[self.hist_len:, :, 6:9]

        x_seq = x_seq.transpose(1, 0, 2)
        y_seq = y_seq.transpose(1, 0, 2)

        x_tensor = torch.from_numpy(np.ascontiguousarray(x_seq, dtype=np.float32))
        y_tensor = torch.from_numpy(np.ascontiguousarray(y_seq, dtype=np.float32))
        return {'x': x_tensor, 'y': y_tensor}

    def stream_and_process(self):
        self._ensure_buffer_ready()

        total_possible_windows = self.total_days - self.seq_len + 1
        if total_possible_windows <= 0:
            return

        start_indices = np.arange(0, total_possible_windows, self.window_stride)
        if self.shuffle_windows:
            rng = np.random.default_rng(self.seed + self.epoch_counter)
            rng.shuffle(start_indices)

        if self.max_windows_per_epoch is not None:
            start_indices = start_indices[:self.max_windows_per_epoch]

        print(
            f"Sampling {len(start_indices):,} windows this epoch "
            f"from {total_possible_windows:,} possible starts "
            f"(stride={self.window_stride}, max_windows={self.max_windows_per_epoch})"
        )
        self.epoch_counter += 1

        for start_t in start_indices:
            yield self._make_sample(int(start_t))

    def __iter__(self):
        return self.stream_and_process()


In [10]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import traceback
import os

try:
    from tqdm.auto import tqdm
except ImportError:
    class tqdm:
        def __init__(self, iterable=None, **kwargs):
            self.iterable = iterable
        def __iter__(self):
            return iter(self.iterable)
        def __enter__(self):
            return self
        def __exit__(self, exc_type, exc, tb):
            return False
        def set_postfix(self, *args, **kwargs):
            pass
        @staticmethod
        def write(message):
            print(message)

# 1. Initialize the Model
# We have 9 input features (6 drivers + 3 history targets), a 3-day forecast, and 3 targets.
# K=2 gives one-hop graph message passing in ChebConv; K=1 is fastest but ignores neighbors.
K_ORDER = 2
model = GLOFPredictor(node_features=9, hidden_channels=32, out_steps=3, out_vars=3, K=K_ORDER)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
model = model.to(device)

print(f"Model initialized on: {device}")

# 2. Use every edge, but move edge_index to the device only once.
edge_index_train = edge_index.detach().contiguous()
print(f"Training graph edges: {edge_index_train.size(1):,} (using all edges)")
edge_index_train = edge_index_train.to(device, non_blocking=True)

# 3. Setup Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 4. Create Dataset and DataLoader
# Use the dummy simulated iterator inside the class block temporarily
parquet_path = r"C:\Users\rubel\Downloads\era5_master_dataset.parquet"# it will ignore since we simulated
# Increase this gradually. Each item is one full graph snapshot; larger batches repeat edge_index internally.
BATCH_SIZE = 1
MAX_WINDOWS_PER_EPOCH = None if MAX_BATCHES_PER_EPOCH is None else MAX_BATCHES_PER_EPOCH * BATCH_SIZE

dataset = SampledGLOFParquetDataset(
    parquet_path,
    node_mapping_path="gnn/node_mapping.csv",
    seq_len=93,
    forecast_len=3,
    cache_path="gnn/era5_buffer_cache.npy",
    window_stride=WINDOW_STRIDE,
    max_windows_per_epoch=MAX_WINDOWS_PER_EPOCH,
    shuffle_windows=SHUFFLE_WINDOWS,
    seed=SAMPLING_SEED,
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, pin_memory=(device.type == "cuda"), num_workers=0)

# 5. Define Loss Function logic
def masked_mse_loss(predictions, targets):
    """
    Computes MSE loss only on valid (non-NaN) target labels.
    predictions: (B, num_nodes, pred_len, out_vars)
    targets: (B, num_nodes, pred_len, out_vars)
    """
    mask = ~torch.isnan(targets)
    if not mask.any():
        return torch.tensor(0.0).to(predictions.device), 0 # No valid labels

    diff = predictions[mask] - targets[mask]
    loss = (diff ** 2).mean()
    valid_count = mask.sum().item()
    return loss, valid_count

print("Beginning Full ST-GNN Training Run...")

def train_model(epochs=10):
    training_logs = []
    
    import os
    os.makedirs("gnn", exist_ok=True)
    
    try:
        with tqdm(range(1, epochs + 1), desc="Epochs", unit="epoch") as epoch_bar:
            for epoch in epoch_bar:
                tqdm.write(f"\n--- Starting Epoch {epoch}/{epochs} ---")
                model.train()
                epoch_loss = 0.0
            
                batch_total = MAX_BATCHES_PER_EPOCH if MAX_BATCHES_PER_EPOCH is not None else None
                batch_bar = tqdm(dataloader, total=batch_total, desc=f"Epoch {epoch}/{epochs}", unit="batch", leave=False)
                for batch_idx, batch in enumerate(batch_bar):
                    x = batch['x'].to(device, non_blocking=True) # Shape: (B, N, seq_len, F)
                    y = batch['y'].to(device, non_blocking=True) # Shape: (B, N, pred_len, out_vars)
                
                    optimizer.zero_grad(set_to_none=True)
                    preds = model(x, edge_index_train)
                    loss, count = masked_mse_loss(preds, y)
                
                    if count > 0:
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        optimizer.step()
                        batch_total_loss = loss.item()
                        total_valid_in_batch = count
                    else:
                        batch_total_loss = 0.0
                        total_valid_in_batch = 0
                
                    avg_batch_loss = batch_total_loss
                    epoch_loss += batch_total_loss
                    batch_bar.set_postfix(loss=f"{avg_batch_loss:.6f}", valid_labels=total_valid_in_batch)
                
                    if batch_idx % 50 == 0:
                        tqdm.write(f"Epoch {epoch} | Dataloader Batch {batch_idx} : Loss = {avg_batch_loss:.6f}, Valid labels = {total_valid_in_batch}")
                    
                    training_logs.append({
                        'epoch': epoch,
                        'batch': batch_idx,
                        'loss': avg_batch_loss,
                        'valid_labels': total_valid_in_batch
                    })
                
                epoch_bar.set_postfix(total_loss=f"{epoch_loss:.4f}")
                tqdm.write(f"--- Epoch {epoch} completed. Total Loss: {epoch_loss:.4f} ---")
                # Save checkpoint at the end of each epoch
                torch.save(model.state_dict(), f"gnn/glof_gnn_ep{epoch}.pth")
                
        # ----------------------------------------
        # Post-Training: Save Final Artifacts
        # ----------------------------------------
        torch.save(model.state_dict(), "gnn/glof_gnn_weights_final.pth")
        print("Final model weights proudly saved to 'gnn/glof_gnn_weights_final.pth'")
        
        if training_logs:
            logs_df = pd.DataFrame(training_logs)
            logs_df.to_csv("gnn/training_logs.csv", index=False)
            print("Training logs saved to 'gnn/training_logs.csv'")
                
    except Exception as e:
        print(f"Training failed. Error: {e}")
        traceback.print_exc()

# Initiate full training for 10 epochs
train_model(epochs=10)

Model initialized on: cuda
Training graph edges: 25,094 (using all edges)
Beginning Full ST-GNN Training Run...


Epochs:   0%|          | 0/10 [00:00<?, ?epoch/s]


--- Starting Epoch 1/10 ---


Epoch 1/10:   0%|          | 0/1500 [00:00<?, ?batch/s]

Loading cached ERA5 buffer from gnn/era5_buffer_cache.npy...
Cached buffer ready: (18628, 6907, 9)
Sampling 1,500 windows this epoch from 18,536 possible starts (stride=7, max_windows=1500)
Epoch 1 | Dataloader Batch 0 : Loss = 0.065629, Valid labels = 62163
Epoch 1 | Dataloader Batch 50 : Loss = 0.002642, Valid labels = 62163
Epoch 1 | Dataloader Batch 100 : Loss = 0.001496, Valid labels = 62163
Epoch 1 | Dataloader Batch 150 : Loss = 0.000987, Valid labels = 62163
Epoch 1 | Dataloader Batch 200 : Loss = 0.000521, Valid labels = 62163
Epoch 1 | Dataloader Batch 250 : Loss = 0.000583, Valid labels = 62163
Epoch 1 | Dataloader Batch 300 : Loss = 0.000425, Valid labels = 62163
Epoch 1 | Dataloader Batch 350 : Loss = 0.000218, Valid labels = 62163
Epoch 1 | Dataloader Batch 400 : Loss = 0.000141, Valid labels = 62163
Epoch 1 | Dataloader Batch 450 : Loss = 0.000104, Valid labels = 62163
Epoch 1 | Dataloader Batch 500 : Loss = 0.000087, Valid labels = 62163
Epoch 1 | Dataloader Batch 550 :

...

Sampling 1,500 windows this epoch from 18,536 possible starts (stride=7, max_windows=1500)
Epoch 8 | Dataloader Batch 0 : Loss = 0.000004, Valid labels = 62163
Epoch 8 | Dataloader Batch 50 : Loss = 0.000001, Valid labels = 62163
Epoch 8 | Dataloader Batch 100 : Loss = 0.000001, Valid labels = 62163
Epoch 8 | Dataloader Batch 150 : Loss = 0.000017, Valid labels = 62163
Epoch 8 | Dataloader Batch 200 : Loss = 0.000003, Valid labels = 62163
Epoch 8 | Dataloader Batch 250 : Loss = 0.000006, Valid labels = 62163
Epoch 8 | Dataloader Batch 300 : Loss = 0.000001, Valid labels = 62163
Epoch 8 | Dataloader Batch 350 : Loss = 0.000002, Valid labels = 62163
Epoch 8 | Dataloader Batch 400 : Loss = 0.000002, Valid labels = 62163
Epoch 8 | Dataloader Batch 450 : Loss = 0.000000, Valid labels = 62163
Epoch 8 | Dataloader Batch 500 : Loss = 0.000000, Valid labels = 62163
Epoch 8 | Dataloader Batch 550 : Loss = 0.000007, Valid labels = 62163
Epoch 8 | Dataloader Batch 600 : Loss = 0.000015, Valid labe

Epoch 9/10:   0%|          | 0/1500 [00:00<?, ?batch/s]

Sampling 1,500 windows this epoch from 18,536 possible starts (stride=7, max_windows=1500)
Epoch 9 | Dataloader Batch 0 : Loss = 0.000001, Valid labels = 62163
Epoch 9 | Dataloader Batch 50 : Loss = 0.000001, Valid labels = 62163
Epoch 9 | Dataloader Batch 100 : Loss = 0.000008, Valid labels = 62163
Epoch 9 | Dataloader Batch 150 : Loss = 0.000003, Valid labels = 62163
Epoch 9 | Dataloader Batch 200 : Loss = 0.000001, Valid labels = 62163
Epoch 9 | Dataloader Batch 250 : Loss = 0.000009, Valid labels = 62163
Epoch 9 | Dataloader Batch 300 : Loss = 0.000003, Valid labels = 62163
Epoch 9 | Dataloader Batch 350 : Loss = 0.000024, Valid labels = 62163
Epoch 9 | Dataloader Batch 400 : Loss = 0.000000, Valid labels = 62163
Epoch 9 | Dataloader Batch 450 : Loss = 0.000003, Valid labels = 62163
Epoch 9 | Dataloader Batch 500 : Loss = 0.000011, Valid labels = 62163
Epoch 9 | Dataloader Batch 550 : Loss = 0.000001, Valid labels = 62163
Epoch 9 | Dataloader Batch 600 : Loss = 0.000010, Valid labe

Epoch 10/10:   0%|          | 0/1500 [00:00<?, ?batch/s]

Sampling 1,500 windows this epoch from 18,536 possible starts (stride=7, max_windows=1500)
Epoch 10 | Dataloader Batch 0 : Loss = 0.000000, Valid labels = 62163
Epoch 10 | Dataloader Batch 50 : Loss = 0.000013, Valid labels = 62163
Epoch 10 | Dataloader Batch 100 : Loss = 0.000004, Valid labels = 62163
Epoch 10 | Dataloader Batch 150 : Loss = 0.000004, Valid labels = 62163
Epoch 10 | Dataloader Batch 200 : Loss = 0.000007, Valid labels = 62163
Epoch 10 | Dataloader Batch 250 : Loss = 0.000000, Valid labels = 62163
Epoch 10 | Dataloader Batch 300 : Loss = 0.000005, Valid labels = 62163
Epoch 10 | Dataloader Batch 350 : Loss = 0.000005, Valid labels = 62163
Epoch 10 | Dataloader Batch 400 : Loss = 0.000006, Valid labels = 62163
Epoch 10 | Dataloader Batch 450 : Loss = 0.000009, Valid labels = 62163
Epoch 10 | Dataloader Batch 500 : Loss = 0.000010, Valid labels = 62163
Epoch 10 | Dataloader Batch 550 : Loss = 0.000001, Valid labels = 62163
Epoch 10 | Dataloader Batch 600 : Loss = 0.00000